# Stemmekloning med Ultimate TTS og OmniVoice
## Elevutgave for Google Colab – norsk, engelsk, latter og sukk

Du laster opp et **10–15 sekunders opptak**, skriver hva som sies i opptaket,
og lager ny tale med samme stemme. Dette er stemmekloning uten trening.
Denne notatboken bruker den opprinnelige OmniVoice-motoren fra Ultimate TTS-appen.

**Alt du trenger av kode og instruksjoner er i denne ene filen.**
Appkode, biblioteker og modellvekter lastes ned automatisk fra offentlige kilder.
Du trenger en Google-konto med tilgang til Colab. Du trenger ikke Hugging Face-konto,
en konto som er 30 dager gammel, API-nøkkel, PRO eller betalingskort.

Tilpasset 9. oktober 2026. Gratis GPU er avhengig av Googles kapasitet og kvoter;
ingen notatbok kan garantere at hele klassen får GPU samtidig.


## Før du begynner

- Ha et tydelig lydopptak på **10–15 sekunder**, helst WAV, MP3 eller M4A, klart på PC-en.
  Spill inn din egen stemme, uten musikk og bakgrunnsprat. Maksimal filstørrelse er 10 MB.
- Skriv ned nøyaktig det du sier. Et opptak og en feil referansetekst kan gi dårlig tale.
- Bruk en norsk referanse til norsk tale, og gjerne en engelsk referanse fra samme person til engelsk.
- Lyden behandles på Googles maskin. Unngå personlige eller fortrolige opplysninger.
  Avspilling i en celle kan bli lagret som del av notatboken: fjern celleutdata før du deler den.
- Bruk resultatet som tydelig merket **KI-generert tale**. Ikke utgi det for et ekte opptak.

**Tjenestevilkår:** Google forbyr «creating deepfakes» i administrerte Colab-økter.
FAQ-en avklarer ikke uttrykkelig hvor grensen går for skoleøvelser med egen stemme;
samtykke er derfor ikke i seg selv en garanti for at Google godtar bruken.
Læreren må vurdere undervisningsopplegget mot disse vilkårene. Notatboken kjører inne
i Colab, siden gratisøkter som hovedsakelig brukes via et separat webgrensesnitt kan avbrytes.
[Googles Colab-FAQ](https://research.google.com/colaboratory/faq.html).

Hvis skolekontoen ikke har tilgang til Colab, må læreren kontakte skolens administrator.


## Åpne filen og velg riktig kjøremiljø

1. Åpne [Google Colab](https://colab.research.google.com/) og logg inn.
2. Velg **File → Open notebook → Upload** (Fil → Åpne notatbok → Last opp),
   og velg `Ultimate_TTS_Colab_Elev.ipynb` fra læreren.
3. Lagre din egen kopi med **File → Save a copy in Drive** hvis du skal beholde notatboken.
4. Velg **Runtime → Change runtime type** (Kjøretid → Endre kjøretidstype).
   Velg **Python 3**, **GPU** og **Runtime Version: 2026.07**. T4 passer hvis det er et eget valg.
   Versjon 2026.07 bruker Python 3.12.13 og er et felles utgangspunkt for klassen.
   [Googles oversikt over kjøremiljøer](https://research.google.com/colaboratory/runtime-version-faq.html).
5. Velg **Save**, og deretter **Connect** øverst til høyre. Et eldre kjøremiljø kan ta litt tid å starte.
6. Kjør kodecellene **én om gangen, i nummerrekkefølge**, ved å trykke ▶ til venstre.
   Vent på at cellen blir ferdig før du går videre. **Ikke velg Run all**:
   du må velge en lydfil og fylle ut tekst underveis.

Hvis Colab ber om betaling eller ikke tildeler GPU, vent eller avtal et nytt tidspunkt med læreren.
Ingen av cellene kjøper ressurser. Ikke opprett ekstra kontoer for å omgå kvoter.


In [ ]:
#@title 1. Kontroller Python og gratis GPU
import platform
import subprocess
import sys

if platform.system() != "Linux" or sys.version_info[:2] != (3, 12):
    raise RuntimeError("Velg Runtime → Change runtime type → Runtime Version 2026.07, "
                       "med Python 3 og GPU. Denne utgaven trenger Python 3.12 på Linux.")
try:
    check = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        check=True, capture_output=True, text=True,
    )
except (FileNotFoundError, subprocess.CalledProcessError):
    raise RuntimeError("Ingen GPU er tildelt. Velg GPU og koble til på nytt. "
                       "Hvis gratis GPU ikke er tilgjengelig, prøv senere.") from None
print("Python:", platform.python_version())
print("GPU:", check.stdout.strip())
print("Kontrollen er ferdig. Kjør celle 2.")


## 2. Installer bibliotekene

Kjør cellen under **én gang per ny økt**. Første gang tar nedlasting og installasjon flere minutter.
La fanen være åpen mens cellen arbeider. Meldinger og fremdrift under cellen er normalt.

Modellens biblioteker installeres i et eget miljø. Du trenger **ikke starte Python på nytt**
etter installasjonen. Hvis du mister økten eller sletter kjøremiljøet, må du begynne på nytt fra celle 1.


In [ ]:
#@title 2. Installer – vent på meldingen «Installasjonen er ferdig»
from pathlib import Path
import platform
import subprocess
import sys

ROOT = Path("/content/ultimate_tts_school")
ENVIRONMENT = ROOT / ".venv"
PYTHON = ENVIRONMENT / "bin/python"
if platform.system() != "Linux" or sys.version_info[:2] != (3, 12):
    raise RuntimeError("Kjør celle 1 og velg kjøremiljø 2026.07 med GPU.")
if "SESSION" in globals() and SESSION.running:
    raise RuntimeError("Avslutt modellen i celle 9 før du installerer på nytt.")
if subprocess.run(["nvidia-smi"], capture_output=True).returncode != 0:
    raise RuntimeError("Ingen GPU tilgjengelig. Kjør celle 1 først.")
ROOT.mkdir(parents=True, exist_ok=True)

def run_install(arguments):
    try:
        subprocess.run(arguments, check=True)
    except subprocess.CalledProcessError as error:
        raise RuntimeError("Installasjonen stoppet. Se den første feilmeldingen over. "
                           "Ved nettverksfeil: kjør celle 2 igjen. Del andre feil med læreren.") from error

print("Installerer støtte for lydfiler ...", flush=True)
run_install(["apt-get", "-qq", "update"])
run_install(["apt-get", "-qq", "install", "-y", "ffmpeg", "libsndfile1", "git"])
if not PYTHON.is_file():
    run_install([sys.executable, "-m", "pip", "install", "--quiet", "virtualenv==20.35.3"])
    run_install([sys.executable, "-m", "virtualenv", "--no-download", str(ENVIRONMENT)])
run_install([str(PYTHON), "-m", "pip", "install", "--upgrade", "pip"])
print("Installerer PyTorch med samsvarende lydbibliotek ...", flush=True)
run_install([str(PYTHON), "-m", "pip", "install",
             "torch==2.8.0", "torchaudio==2.8.0", "torchvision==0.23.0",
             "--index-url", "https://download.pytorch.org/whl/cu128"])
requirements_file = ROOT / "requirements_colab.txt"
requirements_file.write_text('gradio==6.29.1\ntransformers==5.18.0\naccelerate==1.15.0\nhuggingface-hub==1.33.0\nnumpy==2.5.3\nsoundfile==0.14.0\nlibrosa==1.0.0\nsafetensors==0.8.0\nrequests>=2.32.3,<3\nomnivoice @ git+https://github.com/k2-fsa/OmniVoice.git@08be0b4ccbac3e13e374e86fbfead4b4cac343e2\n', encoding="utf-8")
print("Installerer OmniVoice og de øvrige bibliotekene ...", flush=True)
run_install([str(PYTHON), "-m", "pip", "install", "-r", str(requirements_file)])
run_install([str(PYTHON), "-m", "pip", "check"])
print("Installasjonen er ferdig. Kjør celle 3. Du trenger ikke starte Python på nytt.")


## 3. Klargjør skoleutgaven

Denne cellen inneholder de små tilpasningsfilene som notatboken trenger.
Kjør den som den er. Du skal ikke laste opp appfiler, ZIP-filer eller modellfiler selv.
Lærerens appkode og modellversjon er låst slik at elevene bruker samme utgangspunkt.


In [ ]:
#@title 3. Klargjør appen – ingen ekstra filer å laste opp
from pathlib import Path
import importlib
import sys

ROOT = Path("/content/ultimate_tts_school")
PYTHON = ROOT / ".venv/bin/python"
if not PYTHON.is_file():
    raise RuntimeError("Installer først ved å kjøre celle 2.")
if "SESSION" in globals() and SESSION.running:
    raise RuntimeError("Appen kjører allerede. Fortsett med celle 5, eller avslutt i celle 9 først.")
APP_FILES = {'validation.py': '"""Input limits for short classroom voice-cloning demonstrations."""\n\nimport math\nimport re\nfrom pathlib import Path\n\nLANGUAGES = {\n    "Norsk (anbefalt)": "no",\n    "Norsk bokmål": "nb",\n    "Norsk nynorsk": "nn",\n    "English": "en",\n}\nMAX_TEXT_CHARACTERS = 280\nMAX_TEXT_WORDS = 40\n# Documented by the pinned OmniVoice release. Keep the literal model syntax.\nNONVERBAL_TAGS = (\n    "[laughter]", "[sigh]", "[confirmation-en]", "[question-en]",\n    "[question-ah]", "[question-oh]", "[question-ei]", "[question-yi]",\n    "[surprise-ah]", "[surprise-oh]", "[surprise-wa]", "[surprise-yo]",\n    "[dissatisfaction-hnn]",\n)\nBRACKET_TAG = re.compile(r"\\[[^\\[\\]]*\\]")\n\n\ndef validate_tags(text):\n    tags = BRACKET_TAG.findall(text)\n    if any(tag not in NONVERBAL_TAGS for tag in tags):\n        raise ValueError("Ukjent lydtagg. Bruk for eksempel [laughter] eller [sigh], med små bokstaver.")\n    plain = BRACKET_TAG.sub(" ", text)\n    if any(character in plain for character in "[]<>"):\n        raise ValueError("Sjekk klammeparentesene. Bruk bare de støttede OmniVoice-lydtaggene.")\n    return plain, tags\n\n\ndef validate_text_inputs(reference_text, text, language, seed, consent):\n    if consent is not True:\n        raise ValueError("Bekreft at stemmen er din egen, eller at du har tillatelse til å bruke den.")\n    reference_text = str(reference_text or "").strip()\n    text = str(text or "").strip()\n    if not reference_text or len(reference_text) > 600:\n        raise ValueError("Skriv nøyaktig det som sies i opptaket, maksimalt 600 tegn.")\n    plain_text, tags = validate_tags(text)\n    validate_tags(reference_text)\n    spoken_words = len(plain_text.split())\n    if not text or len(text) > MAX_TEXT_CHARACTERS or spoken_words > MAX_TEXT_WORDS:\n        raise ValueError("Skriv en kort tekst eller lydtagg, maksimalt 40 ord og 280 tegn.")\n    if language not in LANGUAGES:\n        raise ValueError("Velg norsk eller engelsk fra språklisten.")\n    try:\n        number = float(seed)\n    except (ValueError, TypeError, OverflowError):\n        raise ValueError("Seed må være et heltall fra null til 2147483647.") from None\n    if not math.isfinite(number) or not number.is_integer() or not 0 <= number <= 2147483647:\n        raise ValueError("Seed må være et heltall fra null til 2147483647.")\n    return {\n        "reference_text": reference_text,\n        "text": text,\n        "language": LANGUAGES[language],\n        "seed": int(number),\n        # Non-verbal sounds also need time; they are not spoken words.\n        "output_seconds": max(2.0, min(18.0, spoken_words / 2.3 + 2.0 * len(tags))),\n    }\n\n\ndef validate_audio_path(path, cache_directory):\n    if not path:\n        raise ValueError("Last opp eller spill inn en referansestemme først.")\n    audio = Path(path).resolve()\n    if not audio.is_relative_to(Path(cache_directory).resolve()) or not audio.is_file():\n        raise ValueError("Bruk et lydopptak lastet opp gjennom appen.")\n    if audio.stat().st_size > 10 * 1024 * 1024:\n        raise ValueError("Referanseopptaket er for stort. Bruk et kort opptak.")\n    return audio\n\n\ndef validate_audio_metadata(frames, sample_rate, channels):\n    if sample_rate <= 0 or channels not in {1, 2}:\n        raise ValueError("Opptaket må ha én eller to lydkanaler og gyldig samplingsfrekvens.")\n    seconds = frames / sample_rate\n    if not 10.0 <= seconds <= 15.0:\n        raise ValueError("Referanseopptaket må vare mellom ti og femten sekunder.")\n    return seconds\n', 'source_loader.py': '"""Load the same application source as the supplied SECourses installer."""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport io\nfrom pathlib import Path, PurePosixPath\nimport sys\nimport zipfile\n\nimport requests\n\nAPP_REVISION = "832d8546bd759c5e9e4ba71e1c42fb9a08aaa329"\nAPP_ARCHIVE_SHA256 = "221ec9786880ec15a5571b6f9d53efdeb5d0e5908a03649dce940a6be8437e8c"\nAPP_ARCHIVE_URL = (\n    "https://codeload.github.com/FurkanGozukara/"\n    f"Premium_IndexTTS2_SECourses/zip/{APP_REVISION}"\n)\nMAX_ARCHIVE_BYTES = 64 * 1024 * 1024\n\n\ndef extract_application(archive: bytes, destination: Path) -> None:\n    """Extract only Python source; reject corrupt or unexpected archives."""\n    if hashlib.sha256(archive).hexdigest() != APP_ARCHIVE_SHA256:\n        raise RuntimeError("Application source checksum mismatch; download rejected.")\n    prefix = f"Premium_IndexTTS2_SECourses-{APP_REVISION}/"\n    with zipfile.ZipFile(io.BytesIO(archive)) as bundle:\n        for member in bundle.infolist():\n            if member.is_dir() or not member.filename.startswith(prefix):\n                continue\n            relative = PurePosixPath(member.filename[len(prefix):])\n            if not relative.parts or relative.parts[0] != "indextts" or relative.suffix != ".py":\n                continue\n            if relative.is_absolute() or ".." in relative.parts or "\\\\" in str(relative):\n                raise RuntimeError("Unsafe application archive path.")\n            target = destination.joinpath(*relative.parts)\n            target.parent.mkdir(parents=True, exist_ok=True)\n            target.write_bytes(bundle.read(member))\n\n\ndef ensure_application(cache_directory: Path) -> Path:\n    destination = cache_directory / APP_REVISION\n    marker = destination / ".verified-source"\n    if not marker.is_file() or marker.read_text(encoding="utf-8") != APP_ARCHIVE_SHA256:\n        print("Downloading pinned Ultimate TTS application source...", flush=True)\n        archive = bytearray()\n        with requests.get(APP_ARCHIVE_URL, stream=True, timeout=(15, 120)) as response:\n            response.raise_for_status()\n            for block in response.iter_content(chunk_size=1024 * 1024):\n                archive.extend(block)\n                if len(archive) > MAX_ARCHIVE_BYTES:\n                    raise RuntimeError("Application source download exceeds the size limit.")\n        extract_application(bytes(archive), destination)\n        required = destination / "indextts/backends/omnivoice.py"\n        if not required.is_file():\n            raise RuntimeError("The application archive does not contain the OmniVoice backend.")\n        marker.write_text(APP_ARCHIVE_SHA256, encoding="utf-8")\n    sys.path.insert(0, str(destination))\n    return destination\n', 'school_engine.py': '"""A classroom adapter around Ultimate TTS\'s original OmniVoiceEngine."""\n\nfrom pathlib import Path\n\nfrom source_loader import ensure_application\n\nMODEL_REVISION = "c5fdb5ccb189668d56333f77ba2629f4cd7535f4"\n\n\ndef create_engine(root: Path, device: str = "cuda:0", gpt_dtype: str | None = None):\n    # app.py imports spaces first: ZeroGPU must patch CUDA before torch imports.\n    ensure_application(root / ".app_source")\n    from huggingface_hub import snapshot_download\n    import torch\n    from indextts.backends.omnivoice import OmniVoiceEngine\n\n    model_directory = root / ".models"\n    snapshot_download(\n        "k2-fsa/OmniVoice",\n        revision=MODEL_REVISION,\n        local_dir=str(model_directory / "omnivoice"),\n        allow_patterns=["*.json", "*.safetensors", "*.jinja", "audio_tokenizer/*"],\n    )\n\n    class SchoolOmniVoiceEngine(OmniVoiceEngine):\n        def _prompt(self, path, settings):\n            # The desktop engine saves voice prompts to disk. This shared demo\n            # builds a prompt per request and does not keep a voice library.\n            if not settings["reference_text"]:\n                raise ValueError("Skriv teksten fra referanseopptaket.")\n            with torch.inference_mode():\n                return self.model.create_voice_clone_prompt(\n                    ref_audio=str(path),\n                    ref_text=settings["reference_text"],\n                    preprocess_prompt=False,\n                )\n\n    engine = SchoolOmniVoiceEngine(\n        model_dir=str(model_directory),\n        runtime={\n            "device": device,\n            "model_variant": "bf16",\n            # Colab selects fp16 for T4, which has no native bf16 support.\n            "gpt_dtype": gpt_dtype or ("bf16" if device.startswith("cuda") else "fp32"),\n            "attention_backend": "sdpa",\n            "use_accel": False,\n            "torch_compile_s2mel": False,\n        },\n    )\n    if not {"no", "nb", "nn", "en"}.issubset(engine.model.supported_language_ids()):\n        raise RuntimeError("The loaded model lacks the requested language IDs.")\n    return engine\n\n\ndef synthesize(engine, payload):\n    return engine.infer(\n        payload["reference_audio"],\n        payload["text"],\n        lang=payload["language"],\n        seed=payload["seed"],\n        omnivoice={\n            "mode": "clone",\n            "reference_text": payload["reference_text"],\n            "num_step": 32,\n            "audio_chunk_duration": 12.0,\n            "audio_chunk_threshold": 18.0,\n            "preprocess_prompt": False,\n        },\n        text_normalization=False,\n        enable_pause_tags=False,\n        # Keep short classroom texts and their sound tags in one context.\n        segmentation_mode="budget",\n        max_text_tokens_per_segment=512,\n        section_batch_size=1,\n        # Bound the amount of GPU work even for unusually slow reference speech.\n        target_duration_s=payload["output_seconds"],\n        target_duration_mode="natural",\n        take_quality_enabled=False,\n    )\n', 'colab_worker.py': 'import json\nimport os\nfrom pathlib import Path\nimport sys\nimport traceback\nimport uuid\n\nROOT = Path(__file__).resolve().parent\nos.environ.setdefault("HF_HOME", str(ROOT / ".hf_cache"))\nos.environ["TOKENIZERS_PARALLELISM"] = "false"\nos.environ["USE_TF"] = "0"\nos.environ["USE_FLAX"] = "0"\nPREFIX = "ULTIMATE_TTS_EVENT "\n\ndef emit(status, **values):\n    print(PREFIX + json.dumps({"status": status, **values}, ensure_ascii=False), flush=True)\n\ntry:\n    import torch\n    import soundfile as sf\n    from school_engine import create_engine, synthesize\n    from validation import validate_audio_metadata, validate_audio_path\n\n    if not torch.cuda.is_available():\n        raise RuntimeError("Ingen GPU tilgjengelig. Velg GPU under Change runtime type.")\n    dtype = "bf16" if torch.cuda.is_bf16_supported() else "fp16"\n    engine = create_engine(ROOT, device="cuda:0", gpt_dtype=dtype)\n    emit("ready", gpu=torch.cuda.get_device_name(0), dtype=dtype,\n         languages=sorted(engine.model.supported_language_ids()))\nexcept Exception as error:\n    traceback.print_exc()\n    emit("error", message=f"Modellen kunne ikke lastes: {type(error).__name__}: {error}")\n    raise SystemExit(1)\n\nfor line in sys.stdin:\n    try:\n        request = json.loads(line)\n        if request["action"] == "close":\n            break\n        if request["action"] != "generate":\n            raise ValueError("Ukjent oppgave.")\n        payload = request["payload"]\n        reference = validate_audio_path(payload["reference_audio"], ROOT / ".audio_cache")\n        info = sf.info(str(reference))\n        validate_audio_metadata(info.frames, info.samplerate, info.channels)\n        sample_rate, pcm = synthesize(engine, payload)\n        output_directory = ROOT / "outputs"\n        output_directory.mkdir(exist_ok=True)\n        output = output_directory / f"KI_tale_{payload[\'language\']}_{uuid.uuid4().hex[:10]}.wav"\n        sf.write(str(output), pcm, sample_rate, subtype="PCM_16")\n        emit("result", path=str(output), seconds=len(pcm) / sample_rate,\n             language=payload["language"], seed=payload["seed"])\n    except Exception as error:\n        traceback.print_exc()\n        if isinstance(error, torch.cuda.OutOfMemoryError):\n            torch.cuda.empty_cache()\n            emit("error", message="GPU-minnet er fullt. Forkort teksten og prøv igjen. "\n                 "Hvis det fortsetter, avslutt modellen i celle 9 og start den igjen i celle 4.")\n        else:\n            emit("error", message=f"{type(error).__name__}: {error}")\n\nengine.unload()\nemit("closed")\n', 'colab_session.py': '"""Notebook-side process control, using only the Python standard library."""\nfrom collections import deque\nimport json\nimport os\nfrom pathlib import Path\nimport queue\nimport subprocess\nimport threading\nimport time\n\nPREFIX = "ULTIMATE_TTS_EVENT "\n\nclass ColabSpeechSession:\n    def __init__(self, root, python):\n        self.root, self.python = Path(root), str(python)\n        self.process = None\n        self.messages = queue.Queue()\n        self.logs = deque(maxlen=25)\n\n    @property\n    def running(self):\n        return self.process is not None and self.process.poll() is None\n\n    def _read(self, process, messages):\n        for line in process.stdout:\n            messages.put(line.rstrip())\n        messages.put(None)\n\n    def _await(self, expected, timeout):\n        deadline = time.monotonic() + timeout\n        next_progress = time.monotonic() + 30\n        while time.monotonic() < deadline:\n            try:\n                line = self.messages.get(timeout=min(2, max(0.01, deadline - time.monotonic())))\n            except queue.Empty:\n                if time.monotonic() >= next_progress:\n                    print("Arbeider fortsatt. Vent til cellen er ferdig.", flush=True)\n                    next_progress = time.monotonic() + 30\n                continue\n            if line is None:\n                raise RuntimeError("Modellprosessen stoppet. Siste meldinger:\\n" + "\\n".join(self.logs))\n            if line.startswith(PREFIX):\n                event = json.loads(line[len(PREFIX):])\n                if event["status"] == "error":\n                    raise RuntimeError(event["message"])\n                if event["status"] == expected:\n                    return event\n            elif line.strip():\n                self.logs.append(line)\n                print(line, flush=True)\n        self.close(force=True)\n        raise TimeoutError("Operasjonen tok for lang tid og ble stoppet. Se feilsøkingen nederst.")\n\n    def start(self):\n        if self.running:\n            print("Modellen er allerede lastet. Fortsett med lydopptak og tekst.")\n            return None\n        self.messages = queue.Queue()\n        self.logs.clear()\n        environment = os.environ.copy()\n        environment["PYTHONUNBUFFERED"] = "1"\n        environment["PYTHONIOENCODING"] = "utf-8"\n        environment.pop("PYTHONPATH", None)\n        self.process = subprocess.Popen(\n            [self.python, "-u", str(self.root / "colab_worker.py")],\n            cwd=str(self.root), env=environment, stdin=subprocess.PIPE,\n            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,\n            text=True, encoding="utf-8", errors="replace", bufsize=1,\n        )\n        threading.Thread(target=self._read, args=(self.process, self.messages), daemon=True).start()\n        try:\n            return self._await("ready", timeout=900)\n        except BaseException:\n            self.close(force=True)\n            raise\n\n    def generate(self, payload):\n        if not self.running:\n            raise RuntimeError("Start modellen ved å kjøre celle 4 først.")\n        try:\n            self.process.stdin.write(json.dumps({"action": "generate", "payload": payload}) + "\\n")\n            self.process.stdin.flush()\n            return self._await("result", timeout=300)\n        except (KeyboardInterrupt, BrokenPipeError):\n            self.close(force=True)\n            raise\n\n    def close(self, force=False):\n        if self.process is None:\n            return\n        process = self.process\n        if process.poll() is None:\n            if not force:\n                try:\n                    process.stdin.write(\'{"action":"close"}\\n\')\n                    process.stdin.flush()\n                    process.wait(timeout=10)\n                except (OSError, subprocess.TimeoutExpired):\n                    force = True\n            if force and process.poll() is None:\n                process.terminate()\n                try:\n                    process.wait(timeout=10)\n                except subprocess.TimeoutExpired:\n                    process.kill()\n                    process.wait(timeout=5)\n        process.stdin.close()\n        if process.stdout is not None:\n            process.stdout.close()\n        self.process = None\n'}
for name, source in APP_FILES.items():
    (ROOT / name).write_text(source, encoding="utf-8")
sys.path.insert(0, str(ROOT))
importlib.invalidate_caches()
from colab_session import ColabSpeechSession
from validation import LANGUAGES, NONVERBAL_TAGS, validate_audio_metadata, validate_audio_path, validate_text_inputs
SESSION = ColabSpeechSession(ROOT, PYTHON)
AUDIO_CACHE = ROOT / ".audio_cache"
AUDIO_CACHE.mkdir(exist_ok=True)
REFERENCE_AUDIO = None
LAST_RESULT = None
print("Skoleutgaven er klar. Kjør celle 4 for å laste modellen.")


## 4. Last modellen

Første modellnedlasting kan ta flere minutter. Vent til det står **«Modellen er klar»**.
En T4 bruker FP16; nyere grafikkort som støtter BF16 velger dette automatisk.
Modellen blir liggende i minnet mellom forsøk, så denne cellen trenger du vanligvis bare én gang.


In [ ]:
#@title 4. Last OmniVoice – kjør én gang per økt
if "SESSION" not in globals():
    raise RuntimeError("Kjør celle 3 først.")
print("Laster modellen. Første nedlasting kan ta flere minutter.", flush=True)
ready = SESSION.start()
if ready:
    print(f"Modellen er klar. GPU: {ready['gpu']}. Presisjon: {ready['dtype']}.")
    print("Språk kontrollert: norsk, bokmål, nynorsk og engelsk.")
print("Fortsett med celle 5.")


## 5. Last opp opptaket ditt

1. Spill inn **10–15 sekunder** tydelig tale. Bruk hele setninger og vanlig snakketempo.
   Ikke ha musikk, lang stillhet eller andre stemmer i bakgrunnen.
2. Kryss av for at du har tillatelse, og kjør cellen under.
3. Velg **én** lydfil fra PC-en. Notatboken lager en mono-WAV som modellen kan lese.
4. Lytt til opptaket som vises. Hvis det er feil opptak, kjør celle 5 igjen.

Et eksempel du kan lese på norsk:
«Hei, dette er et opptak av min egen stemme. Jeg snakker tydelig og i vanlig tempo.
I dag skal vi undersøke hvordan en datamaskin kan lage tale med en stemme som ligner min.»
Tilpass tempo eller antall setninger slik at **det faktiske opptaket** varer 10–15 sekunder.
Et lengre opptak blir avvist, ikke automatisk kuttet til en godkjent referanse.


In [ ]:
#@title 5. Last opp én lydfil på 10–15 sekunder
JEG_HAR_TILLATELSE = False #@param {type:"boolean"}

if "AUDIO_CACHE" not in globals():
    raise RuntimeError("Kjør celle 3 først.")
if JEG_HAR_TILLATELSE is not True:
    raise ValueError("Kryss av: stemmen er din egen, eller personen har gitt tillatelse.")
from google.colab import files
from IPython.display import Audio, display
import os
import subprocess
import uuid
import wave

REFERENCE_AUDIO = None
LAST_RESULT = None
previous_directory = os.getcwd()
try:
    os.chdir(AUDIO_CACHE)
    uploaded = files.upload()
finally:
    os.chdir(previous_directory)
if len(uploaded) != 1:
    raise ValueError("Velg nøyaktig én lydfil. Kjør celle 5 igjen.")
name, data = next(iter(uploaded.items()))
if not data or len(data) > 10 * 1024 * 1024:
    raise ValueError("Bruk én lydfil på maksimalt 10 MB.")
suffix = Path(name).suffix.lower()
if suffix not in {".wav", ".mp3", ".m4a", ".flac", ".ogg", ".webm", ".aac"}:
    raise ValueError("Bruk en lydfil: WAV, MP3, M4A, FLAC, OGG, WebM eller AAC.")
identifier = uuid.uuid4().hex
source = AUDIO_CACHE / f"opplastet_{identifier}{suffix}"
reference = AUDIO_CACHE / f"referanse_{identifier}.wav"
source.write_bytes(data)
# Decode at most 16 seconds: every valid reference is preserved, and longer
# inputs remain over the 15-second limit and are rejected below.
conversion = subprocess.run(
    ["ffmpeg", "-hide_banner", "-loglevel", "error", "-nostdin", "-y", "-i", str(source),
     "-t", "16", "-vn", "-ac", "1", "-ar", "24000", "-c:a", "pcm_s16le", str(reference)],
    capture_output=True, text=True,
)
if conversion.returncode:
    raise ValueError("Lydfilen kunne ikke leses. Prøv et vanlig WAV-opptak. " + conversion.stderr[-600:])
with wave.open(str(reference), "rb") as recording:
    seconds = validate_audio_metadata(recording.getnframes(), recording.getframerate(), recording.getnchannels())
REFERENCE_AUDIO = str(validate_audio_path(reference, AUDIO_CACHE))
print(f"Opptaket er godkjent: {seconds:.2f} sekunder. Lytt, og fyll ut celle 6.")
display(Audio(filename=REFERENCE_AUDIO))
del uploaded, data


## 6. Skriv referansetekst og ny tekst

- **REFERANSETEKST:** Nøyaktig det som sies i opptaket ditt, på språket du faktisk snakker.
  Ikke kopier eksempelet over dersom du har sagt noe annet. Maksimalt 600 tegn.
- **NY_TEKST:** Det du vil at den klonede stemmen skal si. Maksimalt **40 talte ord og 280 tegn**.
  Start med én eller to setninger. Skriv tall og forkortelser slik de skal uttales.
- **SPRÅK:** Språket for den nye talen. «Norsk (anbefalt)» er et godt utgangspunkt.
- **SEED:** Et heltall som styrer variasjonen. Prøv 43 eller 44 hvis første forsøk er dårlig.

**Latter og sukk:** Skriv `[laughter]` og `[sigh]` med små bokstaver i **NY_TEKST**,
også på norsk. De er lydmarkører, ikke ord som skal leses høyt.
Begynn med én tagg. Eksempler:

```text
[laughter] Det var morsomt! [sigh] Nå må vi jobbe videre.
```

```text
[laughter] That was funny! [sigh] Now we need to get back to work.
```

Ikke skriv `[latter]`, `[sukk]`, `[laugh]` eller `[Laughter]`. Taggene teller med i tegngrensen,
og modellen får ekstra tid til å lage lydene. Uttale, stemmelikhet og hvor tydelig taggene høres kan variere.
[OmniVoices dokumentasjon](https://github.com/k2-fsa/OmniVoice/blob/08be0b4ccbac3e13e374e86fbfead4b4cac343e2/README.md#non-verbal--pronunciation-control).


In [ ]:
#@title 6. Fyll ut feltene, og kjør denne cellen
REFERANSETEKST = "" #@param {type:"string"}
NY_TEKST = "Hei! Nå tester vi stemmekloning på norsk. Æ, ø og å skal høres tydelig." #@param {type:"string"}
SPRÅK = "Norsk (anbefalt)" #@param ["Norsk (anbefalt)", "Norsk bokmål", "Norsk nynorsk", "English"]
SEED = 42 #@param {type:"integer"}

if not globals().get("REFERENCE_AUDIO"):
    raise RuntimeError("Last opp og godkjenn referanseopptaket i celle 5 først.")
PAYLOAD = validate_text_inputs(REFERANSETEKST, NY_TEKST, SPRÅK, SEED,
                               globals().get("JEG_HAR_TILLATELSE", False))
PAYLOAD["reference_audio"] = REFERENCE_AUDIO
LAST_RESULT = None
print("Teksten er godkjent. Språk:", SPRÅK, "– seed:", PAYLOAD["seed"])
print("Tekst og lydtagger som sendes til modellen:", PAYLOAD["text"])
print("Kjør celle 7.")


## 7. Lag tale og lytt

Kjør cellen. På en gratis GPU kan et forsøk ta litt tid. Ikke trykk flere ganger mens den kjører.
Vent til lydspilleren vises. Lytt til hele klippet: er alle ordene med, er uttalen forståelig,
og ligner stemmen på opptaket? Korte tekster får en beregnet lydlengde på inntil omtrent 18 sekunder.


In [ ]:
#@title 7. Lag KI-generert tale
import time
from IPython.display import Audio, display

if "SESSION" not in globals() or not SESSION.running:
    raise RuntimeError("Last modellen i celle 4 først.")
if "PAYLOAD" not in globals() or not globals().get("REFERENCE_AUDIO"):
    raise RuntimeError("Kjør celle 5 og celle 6 først.")
# Recheck edited fields and the current file before submitting the job.
payload = validate_text_inputs(REFERANSETEKST, NY_TEKST, SPRÅK, SEED,
                               globals().get("JEG_HAR_TILLATELSE", False))
payload["reference_audio"] = str(validate_audio_path(REFERENCE_AUDIO, AUDIO_CACHE))
LAST_RESULT = None
started = time.perf_counter()
result = SESSION.generate(payload)
LAST_RESULT = result
print(f"Ferdig: {result['seconds']:.1f} sekunder KI-generert lyd. "
      f"Beregningstid: {time.perf_counter() - started:.1f} sekunder.")
print("Språk:", result["language"], "– seed:", result["seed"])
display(Audio(filename=result["path"]))
print("Lytt til resultatet. Last ned med celle 8.")


## 8. Last ned WAV-filen

Kjør cellen under og tillat nettlesernedlasting hvis du blir spurt. Finn filen i Nedlastinger på PC-en.
Notatboken laster ned **det siste vellykkede resultatet**. Gi gjerne filen et navn som viser språk og seed.
Last ned hvert resultat du vil beholde **før** du lager neste forsøk eller avslutter økten.


In [ ]:
#@title 8. Last ned siste KI-genererte WAV
from google.colab import files
if not globals().get("LAST_RESULT"):
    raise RuntimeError("Lag et vellykket resultat i celle 7 først.")
output = Path(LAST_RESULT["path"])
if not output.is_file():
    raise RuntimeError("Filen finnes ikke lenger. Lag lyden på nytt.")
files.download(str(output))
print("Nedlasting startet:", output.name)


## Flere forsøk og elevoppgaven

**Samme stemme, ny tekst:** Endre feltene i celle 6, kjør celle 6, 7 og 8 igjen.
Du trenger ikke installere eller laste modellen på nytt. **Nytt opptak:** Kjør celle 5, 6, 7 og 8.

Gjennomfør disse fire forsøkene, og lagre WAV-filene:

1. Norsk: «Hei! I dag undersøker vi hvordan datamaskinen lager tale. Æ, ø og å skal være tydelige.»
2. Engelsk: «Hello! Today we are testing my cloned voice. The words should be clear and easy to understand.»
   Velg **English**, og bruk gjerne et nytt engelsk referanseopptak fra deg selv.
3. Latter: `[laughter] Det var morsomt!` – velg norsk igjen, og et norsk referanseopptak.
4. Sukk: `[sigh] Nå må vi jobbe videre.`

Lytt etter ord som mangler, uttale, aksent, stemmelikhet og om latter/sukk høres.
Sammenlign to seeds dersom en lyd er svak. Skriv hvilke innstillinger du brukte og hva du observerte.
Ikke hev at et lyduttrykk alltid blir likt eller at alle norske dialekter blir riktige.

Lever filene og en kort refleksjon slik læreren ber om. Merk lyden **KI-generert**.
Hvis du deler selve notatboken, velg **Edit → Clear all outputs** først og kontroller referanseteksten.
Du kan også velge å utelate celleutdata ved lagring under **Edit → Notebook settings**.
Det skal være lærerens valgte deling, ikke automatisk offentlig publisering av stemmen din.


## 9. Avslutt når du er ferdig

Kontroller at WAV-filene er lastet ned. Kjør cellen for å frigjøre modellminnet,
og velg deretter **Runtime → Disconnect and delete runtime** for å slette øktens maskin.
Dette fjerner også de opplastede lydfilene og nedlastede modellene fra kjøremiljøet.
Din lagrede notatbok og lyd du allerede har lastet ned til PC-en slettes ikke.


In [ ]:
#@title 9. Avslutt modellen og frigjør GPU-minnet
if "SESSION" in globals():
    SESSION.close()
print("Modellen er avsluttet. Velg Runtime → Disconnect and delete runtime når du er ferdig.")
print("Vil du fortsette i samme økt? Kjør celle 4 for å laste modellen igjen.")


## Hvis noe ikke virker

| Problem | Hva du gjør |
|---|---|
| Colab er blokkert på skolekontoen | Be læreren kontakte skolens administrator. |
| Ingen gratis GPU / melding om kvote | Vent eller prøv på et senere tidspunkt. Ikke kjøp noe for denne oppgaven. |
| Celle 1 sier feil Python-versjon | Velg **Runtime Version 2026.07**, Python 3 og GPU. Koble til på nytt. |
| Installasjonen stopper | Les første feilmelding. Nettverksfeil: kjør celle 2 igjen. Andre feil: del feilmeldingen med læreren. |
| Modellen bruker tid på nedlasting | Vent til «Modellen er klar». Hold fanen åpen. Ny økt må laste ned filene på nytt. |
| Ingen lydfil / tom referansetekst | Kjør celle 5 og 6, og skriv nøyaktig det som sies i opptaket. |
| Referanseopptaket avvises | Kontroller faktisk lengde 10–15 sekunder og filstørrelse under 10 MB. Prøv WAV. |
| Ukjent lydtagg | Bruk engelske tagger med små bokstaver, for eksempel `[laughter]` og `[sigh]`. |
| Taggen leses som ord eller lyden er svak | Prøv én tagg med en kort setning, et renere opptak og en annen seed. Modellen kan variere. |
| Ord mangler / mumling / feil uttale | Rett referanseteksten, forkort ny tekst og bruk tydelig tale på samme språk. |
| `CUDA out of memory` | Forkort teksten. Ved gjentakelse: kjør celle 9, deretter celle 4. |
| En operasjon stopper etter lang venting | Del siste feilmelding med læreren. Modellen stoppes ved tidsavbrudd; start celle 4 igjen. |
| Du avbrøt celle 4 eller 7 | Modellprosessen avsluttes. Kjør celle 4 for et nytt forsøk. |
| Økten ble koblet fra eller slettet | Nye maskiner mister installasjon og filer. Begynn på nytt fra celle 1. |
| Ingen fil dukker opp i Nedlastinger | Tillat nettlesernedlasting og kjør celle 8 igjen. |

Hvis Google avslutter økten på grunn av tjenestevilkårene, skal du ikke forsøke å omgå blokkeringen.
Ta saken opp med læreren.


## Til læreren

**Prøvekjør før timen:** Installer i en ny Colab-økt, last en kjent referansestemme,
og test norsk, engelsk, `[laughter]` og `[sigh]`. Kontroller at WAV-filene kan lastes ned.
Denne leveransen er kontrollert lokalt for syntaks, selvstendige appfiler, inputgrenser,
taggformidling og prosesskommunikasjon. Ekte Colab-GPU og lydkvalitet er ikke prøvekjørt her.

Notatboken er et interaktivt undervisningsopplegg, ikke en publisert nettside.
Den bruker en egen prosess med et isolert bibliotekmiljø og gjenbruker én lastet modell.
FP16 på T4 unngår krav om BF16-støtte. Ingen Drive-mappe kobles til,
og ingen Hugging Face-token eller betalt API brukes. Modell- og appnedlasting krever nettverk.

Bruk skolens godkjente konto- og personvernopplegg. Google oppgir særskilte krav for
Workspace for Education og mindreårige; kontroller skolens tilgang og nødvendig samtykke
i [Colab-FAQ-en](https://research.google.com/colaboratory/faq.html).
Notatboken dokumenterer Googles forbud mot deepfakes og lover ikke at all stemmekloning godtas.

**Alle dokumenterte lydtagger i denne utgaven:** `[laughter]`, `[sigh]`, `[confirmation-en]`, `[question-en]`, `[question-ah]`, `[question-oh]`, `[question-ei]`, `[question-yi]`, `[surprise-ah]`, `[surprise-oh]`, `[surprise-wa]`, `[surprise-yo]`, `[dissatisfaction-hnn]`.
Skolevalideringen slipper dem gjennom uendret. Vilkårlige pausemarkører og uttalekontroll
med egne parentesuttrykk er ikke aktivert.

**Versjoner:** Original Ultimate TTS-kilde fra revisjon
`832d8546bd759c5e9e4ba71e1c42fb9a08aaa329`, kontrollert med SHA256.
OmniVoice-bibliotek `08be0b4ccbac3e13e374e86fbfead4b4cac343e2` og modell
`c5fdb5ccb189668d56333f77ba2629f4cd7535f4`. PyTorch/torchaudio 2.8.0 og
torchvision 0.23.0 bruker samme CUDA 12.8-distribusjon;
[offisiell kombinasjon](https://pytorch.org/get-started/previous-versions/#v280).
Språk sendes eksplisitt som `no`, `nb`, `nn` eller `en`.
[OmniVoices språkoversikt](https://github.com/k2-fsa/OmniVoice/blob/08be0b4ccbac3e13e374e86fbfead4b4cac343e2/docs/languages.md).

OmniVoice har Apache-2.0-lisens. Dette avklarer ikke lisensen til hele SECourses-premiumappen,
som ikke hadde noen tydelig rotlisens ved kontrollen. Avklar deling/publisering av premiumappen
med opphavspersonen. Notatboken inkluderer våre tilpasningsfiler og henter originalkoden ved kjøring.
[Ultimate TTS-kilden](https://github.com/FurkanGozukara/Premium_IndexTTS2_SECourses).

Colab-miljøer og tjenestevilkår endres over tid. Valgt kjøremiljø 2026.07 er tilgjengelig i en begrenset
periode; oppdater og prøvekjør klassekopien hvis Google fjerner denne versjonen.
